<a href="https://colab.research.google.com/github/fserracrespi/ir2_Project/blob/mhm-dev/pairs/10-gpl/reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Starter: training an encoder

Nobody in the spring course trained an encoder, and about a third of the paper menu asks for it. This notebook runs the loop once, end to end, on a T4 in a few minutes.

1. Score BGE-small on SciFact with the shared testbed.
2. Fine-tune it with `MultipleNegativesRankingLoss`: in-batch negatives, the InfoNCE loss from the recap.
3. Mine hard negatives with the spring cross-encoder and train again.
4. Measure alignment and uniformity (Wang & Isola 2020).
5. Optional: train a LoRA adapter with `peft` instead of the whole model.

**The goal is to see a training loop run, not to beat the baseline.** The positive pairs here are a paper's title and its abstract, taken from the corpus. No test query and no relevance label is used for training: `queries` and `qrels` only appear inside `evaluate`.

On Colab: *Runtime → Change runtime type → T4 GPU*, then run all cells.

In [ ]:
# change 1
import os
if not os.path.exists("testbed.py"):             # on Colab: fetch the repo
    !git clone -q https://github.com/kimgerdes/ir2-testbed
    %cd ir2-testbed
!pip install -q -r requirements.txt datasets accelerate peft

In [ ]:
import random
import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import (CrossEncoder, SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
import testbed

MODEL = "BAAI/bge-small-en-v1.5"
N_TRAIN, N_MINE, BATCH = 2000, 500, 32
random.seed(0)
corpus, queries, qrels = testbed.load("scifact")
docs = [(d["title"], d["text"]) for d in corpus.values() if d["title"] and len(d["text"].split()) >= 50]
random.shuffle(docs)
train_docs, held_out = docs[:N_TRAIN], docs[N_TRAIN:N_TRAIN + 1000]
print(len(corpus), "documents,", len(docs), "usable,", len(train_docs), "for training,", len(held_out), "held out")

## 1. Before training

`evaluate` is the shared testbed: the same search and the same scoring as every row of the results table.

In [ ]:
def evaluate(model, label):
    run = testbed.dense_search(model, corpus, queries, query_prefix=testbed.BGE_QUERY)
    _, ndcg, recall = testbed.score_run(run, "scifact")
    print(f"{label:24} nDCG@10 {ndcg.mean():.3f}  Recall@100 {recall.mean():.3f}")
    return ndcg

model = SentenceTransformer(MODEL, device=testbed.device())
before = evaluate(model, "BGE-small, as downloaded")

## 2. Alignment and uniformity

Alignment: how close the two views of a paper are. Uniformity: how evenly the papers spread over the sphere. Both are better when lower. These are the ten lines of NumPy, on title/abstract pairs the model does not train on.

In [ ]:
def alignment_uniformity(model, pairs):
    a = model.encode([testbed.BGE_QUERY + t for t, _ in pairs], normalize_embeddings=True, batch_size=64)
    b = model.encode([x for _, x in pairs], normalize_embeddings=True, batch_size=64)
    alignment = ((a - b) ** 2).sum(1).mean()
    squared = ((b[:, None] - b[None]) ** 2).sum(-1)[np.triu_indices(len(b), 1)]
    uniformity = np.log(np.exp(-2 * squared).mean())
    return round(float(alignment), 3), round(float(uniformity), 3)

print("alignment, uniformity:", alignment_uniformity(model, held_out))

## 3. Contrastive training with in-batch negatives

Each title must find its own abstract among the abstracts of the batch: with a batch of 32, one positive and 31 negatives. The anchor gets BGE's query prefix, because at test time the query does.

In [ ]:
def train(model, data, output_dir):
    args = SentenceTransformerTrainingArguments(output_dir=output_dir, num_train_epochs=1, per_device_train_batch_size=BATCH,
                                                learning_rate=2e-5, logging_steps=20, save_strategy="no", report_to="none", seed=0,
                                                use_cpu=not torch.cuda.is_available())
    loss = losses.MultipleNegativesRankingLoss(model)
    SentenceTransformerTrainer(model=model, args=args, train_dataset=data, loss=loss).train()

pairs = Dataset.from_dict({"anchor": [testbed.BGE_QUERY + t for t, _ in train_docs], "positive": [x for _, x in train_docs]})
train(model, pairs, "runs/starter")
after = evaluate(model, "after in-batch training")
print("alignment, uniformity:", alignment_uniformity(model, held_out))

## 4. Hard negatives from the spring cross-encoder

Random abstracts are easy to reject. So for each title we take the current model's top 20 abstracts, ask the cross-encoder which wrong one looks most relevant, and add it to the loss as an extra negative.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=testbed.device())
abstracts = [x for _, x in train_docs]
D = model.encode(abstracts, normalize_embeddings=True, batch_size=64)
triples = {"anchor": [], "positive": [], "negative": []}
for i, (title, abstract) in enumerate(train_docs[:N_MINE]):
    q = model.encode(testbed.BGE_QUERY + title, normalize_embeddings=True)
    wrong = [j for j in np.argsort(-(D @ q))[:21] if j != i][:20]
    scores = cross_encoder.predict([[title, abstracts[j]] for j in wrong], show_progress_bar=False)
    triples["anchor"].append(testbed.BGE_QUERY + title)
    triples["positive"].append(abstract)
    triples["negative"].append(abstracts[wrong[int(np.argmax(scores))]])

i = 0
print("title:        ", train_docs[i][0])
print("hard negative:", triples["negative"][i][:160], "…")
train(model, Dataset.from_dict(triples), "runs/starter")
hard = evaluate(model, "after hard negatives")

## 5. Did it change anything?

The same test as in the results table: a paired t-test and a bootstrap interval over the 300 queries. A small difference with a large p-value is noise, not a win.

In [ ]:
print("in-batch vs original:      ", testbed.compare(after, before))
print("hard negatives vs original:", testbed.compare(hard, before))

## 6. Optional: a LoRA adapter

Instead of updating all the weights, train a small adapter on top of frozen ones. Set `USE_LORA = True`. Tested with BGE-small; another architecture may need `target_modules` in `LoraConfig`.

In [ ]:
USE_LORA = False
if USE_LORA:
    from peft import LoraConfig, TaskType
    lora_model = SentenceTransformer(MODEL, device=testbed.device())
    lora_model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
                                      r=8, lora_alpha=16, lora_dropout=0.1))
    train(lora_model, pairs, "runs/starter")
    lora = evaluate(lora_model, "after LoRA training")
    print("LoRA vs original:", testbed.compare(lora, before))

## From here to your testbed row

1. Keep your model: `model.save("runs/my-model")`.
2. Start from `pairs/_template/method.py`: it writes one run per dataset to `runs/`.
3. Score it: `python testbed.py score pairs/<your folder>/runs --method "<method>" --pair <your folder> --against <closest baseline>`.